# Experimental Descriptive Research

**Topic 06 · 2 lectures**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb06_experimental_descriptive_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** description (descriptive kind · data-at-hand reach). A
**descriptive question** asks what is actually there in the cases you can see. This
week the "what" is a **latent characteristic**, a real property you cannot read off
directly, like how much discrimination a job market holds or how sensitive an
attitude is to how it is framed. You use a randomized stimulus as a measuring
device to reveal it, and you describe what you find for the units you observed.

**Design pathway:** experimental descriptive. **Experimental** means you assign a
**controlled stimulus** by chance: you decide, at random, which version each
respondent sees. The surprise of this week is that random assignment does not force
a causal question. When the target is a latent characteristic and not the effect of
an intervention you would deploy, the inquiry stays descriptive.

| | |
|---|---|
| **A claim this topic PERMITS** | "Using randomized stimuli as a measurement instrument, the latent characteristic I estimate is [value] for the units observed, with named threats to construct validity and stated uncertainty." |
| **A claim this topic does NOT permit** | "We randomized, therefore we have a causal effect," or a measurement number reported without checking whether participants guessed the hypothesis or the instrument itself moved the result. |

**Where this sits in the course:** Week 7 of the design library. It builds on your
observational descriptive audit (M4) and your causal-language boundary (M5), and it
develops **M6, your experimental measurement or data-acquisition protocol**, which
you present at this week's Friday studio. M6 then hands off to M7, the declared
analysis protocol.

*Provenance: RDSS ch. 17 'Experimental: descriptive' + the DeclareDesign experimental-descriptive design library + rdss bonilla_tillery | experiments as measurement systems, latent characteristics, controlled stimuli, audit + list + conjoint experiments, construct validity, demand and instrument effects, assignment != causal inquiry | a real survey experiment read as measurement + a seeded list-experiment + a seeded measurement simulation (seed 464) | adapted (R->Python, honors non-quant audience)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how an experiment can serve as a **measurement instrument** for a
   **latent characteristic**, and why random assignment alone does not make an
   inquiry causal.
2. Read a real survey experiment as a descriptive measurement: the mean outcome
   under each controlled stimulus, described for the units observed, with its
   uncertainty.
3. Recover a hidden **prevalence** with a seeded **list experiment**, and say what
   descriptive quantity it estimates that a direct question cannot.
4. Diagnose **demand effects** and **instrument effects** as threats to **construct
   validity**, by seeded simulation.
5. Declare, diagnose, and **redesign** a small measurement experiment to shrink an
   artifact, verifying each redesign by simulation.
6. Apply experiments-as-measurement to your own project's **M6** experimental
   measurement or data-acquisition protocol.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the measurement-experiment DAGs; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you work with **Gemini** as a research partner, not an answer machine. A
good use looks like this: you commit your own answer first, then ask Gemini to
locate a real study you verify, explain a result you check against the printout, or
attack a caveat you wrote. A bad use is letting it decide whether your inquiry is
descriptive or causal, or hand you a "measurement" number with no check on what
biased it.

**Never delegate these:** whether an experiment truly measures your construct,
whether your inquiry is descriptive or causal, and which artifacts you must rule
out. Those are yours to declare and defend. The full list of never-delegate
decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own answer
before you open Gemini. Commit your work to git before a long AI session too, so
you can always compare what you wrote against what the tool changed.

> **A question that often comes up here:** *"An experiment sounds causal by
> definition. Why is this week filed under description?"* Because the word that
> decides kind is your **question**, not your **procedure**. Random assignment is a
> data-strategy tool. You can point it at a causal question or at a descriptive one.
> When your target is a property of the world as it stands, a latent characteristic
> you are trying to measure, the inquiry is descriptive even though you shuffled.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A famous study mails out thousands of nearly identical job résumés. The only thing
that changes is the name at the top, randomly assigned: some résumés carry a name
that signals one racial group, others a name that signals another. The researchers
count callbacks and find the two name groups get called back at clearly different
rates.

Here is the claim on the table: **"Because the names were randomly assigned, this
study estimates a causal effect."** Would you stake your name on it, yes or no?
Commit to one answer before we go further. Then defend it. What single quantity is
this study actually built to report to the world: the effect of changing a name, or
the amount of discrimination sitting in this labor market right now? If it is the
second, then the random assignment was a measuring trick, not a causal claim.
Naming that difference is the whole job of Lecture 1.

## 1. Experiments as Measurement Systems

**Guiding question:** *if you randomize a stimulus but only want to measure
something that is already there, is your inquiry causal or descriptive?*

> *"Show me the number your design is built to report. If that number is a property
> of the world as it stands, you measured something. Do not dress it up as a cause
> just because you flipped a coin to reveal it."*
> — a **thesis advisor** reading the methods section for the first time

Three terms carry this whole section. Meet them one at a time.

- **Latent characteristic**: a real property you cannot read off directly, so you
  have to reveal it through a design. Example: the amount of hiring discrimination
  in a job market. No one prints it on a form. You have to expose it.
- **Controlled stimulus**: a version of a prompt, résumé, or question that you set
  on purpose and assign by chance, so that any difference in response can be pinned
  to the version rather than to who received it. Example: two résumés identical
  except for the name at the top.
- **Experiment as a measurement system**: using randomly assigned, controlled
  stimuli as an instrument to reveal a latent characteristic. Example: randomizing
  résumé names to read off the level of discrimination the market applies.

The résumé study is an **audit experiment**. An **audit experiment** sends
matched, randomly varied applications or requests into the real world to measure
how much a group is treated differently. The random assignment of names is not there
to recommend "rename your applicants." It is there to isolate one signal so the
callback gap becomes a clean measurement of discrimination. The estimand, the exact
quantity the study is built to report, is a description of the world: how much
discrimination exists, for the units observed.

Picture the machinery as a small diagram. This course draws designs as a **DAG**,
short for directed acyclic graph, a picture of arrows where each arrow means "this
influences that." The next cell draws the measurement-experiment DAG so you can see
why the inquiry stays descriptive.

The one idea to hold: the randomized stimulus feeds the measured response, but the
**target of the inquiry** is the latent characteristic behind it. You are pointing
an instrument at something that already exists.

In [ ]:
# The measurement-experiment DAG: a randomized stimulus reveals a latent trait.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Randomized\nstimulus (Z)":     (0.0, 0.0),
    "Latent\ncharacteristic\n(theta)": (0.0, 1.6),
    "Measured\nresponse (Y)":       (2.4, 0.8),
    "Demand /\ninstrument\nartifact (A)": (2.4, 2.2),
}
edges = [
    ("Randomized\nstimulus (Z)", "Measured\nresponse (Y)"),
    ("Latent\ncharacteristic\n(theta)", "Measured\nresponse (Y)"),
    ("Demand /\ninstrument\nartifact (A)", "Measured\nresponse (Y)"),
]
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(9.5, 5))
# The latent-characteristic node is the TARGET of inquiry: give it a bold outline.
node_colors = ["#DCE6F1", "#FCE4B8", "#DCE6F1", "#EAEAEA"]
node_edges  = ["#4C72B0", "#B8860B", "#4C72B0", "#888888"]
node_widths = [1.5, 3.0, 1.5, 1.5]
nodes = list(pos.keys())
for n, fc, ec, lw in zip(nodes, node_colors, node_edges, node_widths):
    nx.draw_networkx_nodes(G, pos, nodelist=[n], node_color=fc, edgecolors=ec,
                           linewidths=lw, node_size=6200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8.5, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=18, edge_color="#555555",
                       width=1.6, node_size=6200, ax=ax)
ax.text(0.0, 2.55, "TARGET OF THE INQUIRY:\na summary of theta (descriptive)",
        ha="center", va="bottom", fontsize=9, color="#B8860B", fontweight="bold")
ax.set_title("A measurement experiment: the stimulus reveals a latent trait you describe")
ax.set_xlim(-1.3, 3.7)
ax.set_ylim(-0.7, 3.1)
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Measurement-experiment DAG drawn.")
print("  Z -> Y exists, but the target of the inquiry is theta, the latent trait.")
print("  Because you describe theta (not the effect of deploying Z), the kind is DESCRIPTIVE.")
print("  Node A previews Lecture 2: artifacts (demand + instrument) also feed Y and can fake theta.")

**Reading the diagram.** Follow the arrows into the measured response. Two honest
sources feed it: the randomized stimulus and the latent characteristic you care
about. The stimulus is the flashlight. The latent characteristic is the thing in the
dark you are trying to see. Your estimand is a summary of that latent characteristic,
which is a **descriptive** target, not the effect of an intervention you would roll
out.

The highlighted node names the whole point: the target of the inquiry is the latent
trait, so the kind is descriptive. The gray node on top is a warning for Lecture 2.
Artifacts can also feed the measured response and fake the trait, which is why the
hue differs and carries no truth value.

*In plain terms, the picture says a coin flip in your design does not decide whether
your question is descriptive or causal. What you are trying to learn decides that.*

> **A question that often comes up here:** *"If the callback gap is caused by the
> name, isn't that a causal finding?"* There is a real causal effect of the name on
> callbacks inside the study, yes. But that effect is a **tool**, not the point. No
> one wants to know "what happens if we rename applicants." They want to know how
> much discrimination the market holds. The experiment converts a causal contrast
> into a **measurement** of a standing feature of the world. The reported estimand
> is descriptive, and calling it a causal effect of a deployable intervention
> misreads the question.

**🏠 Homework depth — run this on your own before the next lecture.**
**Before you ask:** in one sentence, name a real audit or list experiment you think
exists in your field. Write the name first, so you can see whether the tool confirms
a real one or invents a convincing fake.

> 💡 **Gemini Prompt:** "Act as a literature scout. List three real, published
> experiments that were used to MEASURE a latent characteristic rather than to test a
> deployable intervention: audit studies of discrimination, list experiments for
> sensitive attitudes, or conjoint studies of preferences. For each, give authors,
> year, venue, and the one latent quantity it measured, in a table. Only include work
> you are confident exists, and mark any row you are unsure about. Then tell me which
> single row you are least certain is real, and exactly what I would search to
> confirm it."
>
> **After running, verify (counters *confident fabrication*: a fluent citation with
> real-sounding authors and a real journal for a paper that does not exist):**
> - [ ] Open a library or scholar search and retrieve each study yourself. Any row
>   you cannot find in a minute is fabricated until proven otherwise, so cut it.
> - [ ] For each surviving row, confirm the study measured a latent characteristic,
>   not the effect of a rollout. A mislabeled row is a scope error, not a source.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

## 2. A Survey Experiment as a Measurement Instrument

**Guiding question:** *once the stimulus is assigned, what descriptive quantity does
the experiment let you report, and where must you stop?*

> *"Every table you show me answers a question about some quantity. Before you tell
> me a frame moves opinion, show me the number the design measures, and show me its
> uncertainty. A gap smaller than its own noise is not a finding."*
> — a **journal reviewer** on the first draft they will read

You will work with a real survey experiment. **bonilla_tillery** is a replication of
a study that randomly assigned respondents to read one of four identity **frames**
for a social movement, then recorded their support on a 0-to-1 scale. A **survey
experiment** embeds a randomly assigned stimulus inside a survey, so the survey
itself becomes the measuring instrument.

Here the four frames are the **controlled stimulus** (the randomized `Z`), and
support is the measured response. Read descriptively, the experiment measures two
things at once: the **level** of support in these respondents, and how **stable**
that support is when you change the words around it. A **stable** attitude barely
moves across frames. A **fragile** one swings. Both are descriptions of the units
you observed, not recommendations to deploy a frame.

### 🔮 Pause & Predict

You are about to compute mean support under each of the four randomly assigned
frames, on a 0-to-1 scale. Before running the next cell, commit: will the four frame
means land far apart (a fragile, easily-moved attitude) or close together (a stable,
crystallized one)? Name a rough gap you expect between the highest and lowest frame,
in points on the 0-to-1 scale. Write one sentence of reasoning. Do not peek at the
output first. You are predicting a reveal.

### YOUR ANSWER HERE:

**Far apart or close together (my predicted high-minus-low gap):**

**One sentence of reasoning:**

---

### 🛠️ Run the Study: measure support under each frame

Run the cell below. It reports mean support under each randomly assigned frame, with
each frame's standard error (a **standard error** is the typical distance between a
sample mean and the truth it estimates, so a gap smaller than the standard errors is
inside the noise). It then plots the four frame means with error bars. Read the
spread before you read the next markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting whether Gemini will keep the reading
descriptive (a measurement of whether framing moves the attitude), or (wrongly) upgrade
it to "the general frame causes more support."

> 💡 **Gemini Prompt:** "Here is a cell that computes mean support under four randomly
> assigned identity frames and prints each frame's mean and standard error: [paste the
> next cell]. Explain, line by line, what the groupby computes. Then state, in one
> sentence, what descriptive quantity this measures. Finally, tell me whether the
> gap between the highest and lowest frame is large relative to the standard errors,
> and what that implies about how framing moves this attitude."
>
> **After running, verify (counters *silent scope change*: an AI that quietly upgrades
> 'support is high and frame-stable' into 'the general frame CAUSES more support' has
> answered a causal question you did not ask):**
> - [ ] Read the answer's claim next to your question, word for word. If it says a
>   frame "causes," "boosts," or "should be used," reject it. Your inquiry is
>   descriptive: it measures the attitude, it does not prescribe a frame.
> - [ ] Confirm the frame means and the high-minus-low gap it quotes match the numbers
>   your cell printed, not numbers it guessed. Compare the gap to the standard error of
>   the *difference* (roughly 0.022 here), not one frame's 0.015: a gap under about two
>   of those is inside the noise, which is why this one names no winning frame.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Load the survey experiment and confirm its shape before trusting anything.
bt = load_course_data("bonilla_tillery.csv")
assert bt.shape == (849, 10), "unexpected shape — flag this!"
print("✓ Loaded bonilla_tillery.csv —", bt.shape[0], "rows,", bt.shape[1], "columns")
print("  Z = the randomly assigned identity frame; blm_support = support on a 0-to-1 scale.")
print()

# Measure: mean support under each randomly assigned frame, with a standard error.
by_frame = bt.groupby("Z")["blm_support"].agg(["mean", "std", "count"])
by_frame["se"] = by_frame["std"] / np.sqrt(by_frame["count"])
by_frame = by_frame.round(3)
print("Mean support under each controlled frame (the measurement):")
print(by_frame[["mean", "se", "count"]].to_string())

overall = bt["blm_support"].mean()
hi, lo = by_frame["mean"].max(), by_frame["mean"].min()
se_gap = float(np.sqrt(by_frame["se"].max()**2 + by_frame["se"].min()**2))
print(f"\n  overall mean support = {overall:.3f}")
print(f"  highest frame mean   = {hi:.3f}   lowest frame mean = {lo:.3f}")
print(f"  high-minus-low gap   = {hi - lo:.3f}   (about {(hi - lo) / se_gap:.1f} standard errors of that gap, inside the noise)")

fig, ax = plt.subplots(figsize=(8.5, 5))
order = by_frame.sort_values("mean").index
means = by_frame.loc[order, "mean"].values
ses   = by_frame.loc[order, "se"].values
ax.errorbar(range(len(order)), means, yerr=1.96 * ses, fmt="o", color="#4C72B0",
            capsize=6, markersize=9, linewidth=1.5, label="frame mean ± 95% interval")
ax.axhline(overall, color="#666666", linestyle="--", linewidth=1.5,
           label=f"overall mean = {overall:.2f}")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order)
ax.set_ylim(0.70, 0.92)
ax.set_ylabel("Mean support for the movement (0 to 1)")
ax.set_xlabel("Randomly assigned identity frame")
ax.set_title("Measuring a crystallized attitude: support is high and barely moves across frames")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: support is high everywhere, and the frame gap is inside the noise.
gap = by_frame["mean"].max() - by_frame["mean"].min()
se_of_gap = float(np.sqrt(by_frame["se"].max()**2 + by_frame["se"].min()**2))
assert by_frame["mean"].min() > 0.79, "every frame should read as high support"
assert gap < 0.05, "frames move support only a little"
assert gap < 2 * se_of_gap, "the biggest gap should be within about two standard errors"
print(f"✓ Self-check passed: support runs {by_frame['mean'].min():.3f} to "
      f"{by_frame['mean'].max():.3f} across frames;")
print(f"  the biggest gap ({gap:.3f}) is about {gap / se_of_gap:.1f} standard errors, inside the noise.")
print("  The measurement: a HIGH, frame-STABLE attitude in these respondents.")

### 🔍 Reading the Evidence: what this measures, and what it forbids

The four frames all land high, between about 0.80 and 0.84 support, and the biggest
gap is roughly 0.04, which is about 1.7 standard errors of that gap, under the usual
two. Read as measurement,
the experiment reports a **crystallized attitude**: support is high and barely
shifts when you reword the appeal. That is a genuine descriptive finding about the
units observed, and it is more interesting than any single average, because it tells
you the attitude is settled, not up for grabs.

In the cell below, write three things. First: the honest descriptive headline this
measurement supports, worded for the units observed. Second: the tempting causal
sentence you must NOT write, the one that treats the small frame gap as "the general
frame works." Third: name why random assignment here served a descriptive inquiry
rather than a causal one.

> **A question that often comes up here:** *"The general frame is highest. Why not
> just say it wins?"* Because "wins" is a causal, deployable claim, and the gap that
> would support it is smaller than its own uncertainty. The design measured whether
> support is frame-stable, and the answer is that it is. Reporting the noise as a
> winner is the exact overreach this week exists to catch.

### YOUR ANSWER HERE:

**My honest descriptive headline (for the units observed):**

**The causal sentence I must NOT write:**

**Why random assignment here served a descriptive inquiry:**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 1's in-class block.** Everything below deepens
the same ideas. Finish it on your own before the next lecture: the list-experiment
simulation, its two homework-depth Gemini prompts, and the design choice between an
audit experiment, a list experiment, and a direct question. Come back with a first
note on whether an experiment could measure a construct in your own project.

---

## 3. The List Experiment: Measuring a Hidden Prevalence

**Guiding question:** *when a direct question makes people hide the truth, how can
random assignment measure the truth anyway, and stay purely descriptive?*

Some things people will not admit to an interviewer. Ask them directly and you get
**sensitivity bias**, the gap between what people truly hold and what they are
willing to report on a sensitive item. A **list experiment** measures the hidden
**prevalence**, the share of people who hold the sensitive attitude, without ever
asking anyone to admit it.

Here is the trick. You randomly assign respondents to two groups.

- The **control** group sees a list of ordinary, non-sensitive items and reports only
  **how many** they agree with, not which ones.
- The **treatment** group sees the same list plus one sensitive item, and again
  reports only the count.

Because assignment is random, the two groups are alike on average. So the difference
in their average counts estimates the share who agree with the sensitive item.
Nobody ever points to the sensitive item. The estimand is a **prevalence**, a plain
description of how common an attitude is. Random assignment is essential to the
measurement, yet the inquiry never leaves description.

### 🔮 Pause & Predict

The next cell builds a list experiment where the sensitive attitude is truly held by
**40% of people**, but a direct question makes many of them hide it. Before running,
commit to two predictions. First: will the list-experiment estimate land near 0.40,
or far off? Second: will a direct question land above, at, or below the true 0.40,
and why? Write one sentence. Do not run the cell until you have committed.

### YOUR ANSWER HERE:

**List-experiment estimate (near 0.40 or not):**

**Direct question (above / at / below 0.40, and why):**

---

### 🛠️ Hands-On: recover a hidden prevalence

Run the cell. It plants a true sensitive-item prevalence of 0.40, randomly assigns
respondents to the control list or the treatment list, and estimates the prevalence
from the difference in average counts. It then shows what a direct question would
have reported instead.

**🏠 Homework depth: run this on your own.**
**Before you ask:** in one sentence, write down whether "difference in average counts"
is the right estimator here, and one assumption it needs.

> 💡 **Gemini Prompt:** "Here is a list-experiment simulation that estimates a hidden
> prevalence as the difference in mean item-counts between a treatment list and a
> control list: [paste the next cell]. Name the assumptions this difference-in-means
> estimator relies on for the estimate to be valid: for example, that adding the
> sensitive item does not change how people answer the other items, and that
> assignment is truly random. For each assumption, tell me one way it could fail in a
> real survey."
>
> **After running, verify (counters *plausible-but-wrong-method*: an estimator that
> sounds standard can be wrong when its assumptions do not hold for your data):**
> - [ ] Confirm each assumption Gemini names actually applies to a list experiment,
>   and check that the estimate it quotes matches the prevalence your cell printed
>   (near 0.44) and its interval, not a number it guessed.
> - [ ] Ask a follow-up: "which single assumption, if broken, would most bias the
>   estimate, and in which direction?" A method critique with no failure direction is
>   incomplete.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# A list experiment recovers a hidden prevalence via random assignment.
# Re-seed a fresh generator so this result is reproducible on its own (seed 464).
rng_list = np.random.default_rng(SEED)
N = 2000
J_prev = [0.55, 0.40, 0.65, 0.30]          # four ordinary, non-sensitive baseline items
TAU_TRUE = 0.40                            # the TRUE prevalence of the sensitive attitude

baseline = np.column_stack([rng_list.random(N) < p for p in J_prev]).astype(int)
holds_sensitive = (rng_list.random(N) < TAU_TRUE).astype(int)   # hidden truth
treat = rng_list.random(N) < 0.5                                # random assignment

reported = np.where(treat, baseline.sum(axis=1) + holds_sensitive, baseline.sum(axis=1))
mean_T, mean_C = reported[treat].mean(), reported[~treat].mean()
est_prev = mean_T - mean_C
sT, sC = reported[treat].std(ddof=1), reported[~treat].std(ddof=1)
se = np.sqrt(sT**2 / treat.sum() + sC**2 / (~treat).sum())

# A direct question: only some holders will admit the sensitive attitude out loud.
admits = holds_sensitive & (rng_list.random(N) < 0.55)
direct_prev = admits.mean()

print(f"Random assignment: {treat.sum()} treatment, {(~treat).sum()} control")
print(f"  mean count, treatment list = {mean_T:.3f}")
print(f"  mean count, control list   = {mean_C:.3f}")
print(f"  list-experiment estimate   = {est_prev:.3f}  (95% interval "
      f"[{est_prev - 1.96*se:.3f}, {est_prev + 1.96*se:.3f}])")
print(f"  TRUE prevalence planted     = {TAU_TRUE:.3f}")
print(f"  direct-question estimate    = {direct_prev:.3f}  (under-reports by "
      f"{TAU_TRUE - direct_prev:+.3f})")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["List\nexperiment", "Direct\nquestion"], [est_prev, direct_prev],
              color=["#4C72B0", "#DD8452"], edgecolor="white")
ax.axhline(TAU_TRUE, color="#666666", linestyle="--", linewidth=2,
           label=f"true prevalence = {TAU_TRUE:.2f}")
ax.set_ylim(0, 0.55)
ax.set_ylabel("Estimated share holding the sensitive attitude")
ax.set_title("The list experiment recovers the hidden prevalence; the direct question hides it")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the list experiment brackets the truth; the direct question sinks below it.
assert est_prev - 1.96*se <= TAU_TRUE <= est_prev + 1.96*se, "true prevalence should sit inside the interval"
assert direct_prev < TAU_TRUE - 0.10, "the direct question should badly under-report"
print(f"✓ Self-check passed: the list-experiment interval "
      f"[{est_prev - 1.96*se:.3f}, {est_prev + 1.96*se:.3f}] covers the true 0.40;")
print(f"  the direct question ({direct_prev:.3f}) misses it by more than 0.10. Same people, honest vs shy.")

### 🔍 Reading the Evidence: what the two numbers mean

The list experiment lands near 0.44, and its interval covers the true 0.40. The
direct question reports about 0.22, roughly half the truth, because shy holders will
not admit the sensitive item to an interviewer. Both numbers are **descriptive**:
each is an estimate of a prevalence, a share of people. Neither claims that anything
**causes** the attitude. The random assignment did one job: it made the treatment and
control lists comparable so their count difference could stand in for the hidden
share.

The list experiment costs you precision. Its interval is wide, because you are
reading a small signal out of noisy counts. That is the trade: it buys honesty about
a sensitive quantity at the price of a wider uncertainty band. Reporting the estimate
without that band would repeat the overclaiming-certainty failure this course keeps
naming.

### YOUR ANSWER HERE:

**Which estimate is closer to the truth, and why the other one misses:**

**One sentence stating why both estimates are descriptive, not causal:**

---

### ⚖️ Make a Design Choice: audit, list, or direct question?

*(Human-first: settle your own choice and its defense before you ask any AI.)*

You want to measure how common a stigmatized behavior is among a group, for the
descriptive question *what share of people do this?* You have three designs. **Choose
one and defend it** in a short paragraph, naming what each one measures and what
threatens it:

- **A.** A **direct question** that simply asks each person whether they do it.
- **B.** A **list experiment** that hides the sensitive item inside a count.
- **C.** An **audit experiment** that sends matched requests into the world and
  records how differently the group is treated.

In the cell below, pick A, B, or C, then defend it: what descriptive quantity does
your choice report, and what is its worst threat?

### YOUR ANSWER HERE:

**My chosen design (A / B / C):**

**What it measures, and its worst threat:**

**Why my inquiry stays descriptive under this design:**

---

### 📝 Practice: descriptive measurement or causal effect?

*(Human-first: classify all three yourself before any AI. Reading the estimand off
the question's words is the graded skill.)*

Each study below randomly assigns a stimulus. For each, decide whether the inquiry
is **descriptive (a measurement)** or **causal (the effect of an intervention someone
would deploy)**, and justify the call from the exact quantity the question wants.
Answer in the cell that follows.

- **A.** A team randomizes minority-signaling names on identical rental inquiries and
  reports the callback gap as *the level of housing discrimination in this city*.
- **B.** A company randomly gives half its users a new checkout button and compares
  purchase rates to decide *whether launching the button will raise sales*.
- **C.** A survey randomly assigns respondents to a list-experiment treatment or
  control to estimate *what share of residents privately hold a stigmatized view*.

### YOUR ANSWER HERE:

**A (descriptive or causal, and the quantity the question wants):**

**B (descriptive or causal, and the quantity the question wants):**

**C (descriptive or causal, and the quantity the question wants):**

---

You have built the core idea: a randomized stimulus can be an instrument that
measures a latent characteristic, and the inquiry stays descriptive. Lecture 2 turns
to the harder half. A measurement is only as good as its freedom from **artifacts**,
the demand effects and instrument effects that can fake a result, and you will
declare, diagnose, and redesign a measurement experiment to shrink them.

---

---

# Lecture 2

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

Two labs are handed the **same** sealed sensor and asked to calibrate it against the
same reference standard. Lab 1 reports the sensor reads 2% high. Lab 2 reports it
reads 2% low. The sensor did not change between labs, and both teams are careful.

Here is the question on the table: the two readings disagree, so **where does the
disagreement live**, in the world, in the instrument, or in the design of the
measurement? Commit to one answer before we go further. Then defend it. If the thing
being measured is fixed and the instrument is one sealed unit, what is left that could
push two honest measurements apart? Naming that third source, and learning to design
it out, is the whole job of Lecture 2.

## 4. When the Measurement Is an Artifact

**Guiding question:** *how do you tell a real measurement from a number your own
design manufactured?*

> *"I do not doubt your instrument read what it read. I doubt that what it read is the
> thing you meant to measure. Show me you ruled out the two ways your setup could have
> produced this number on its own."*
> — a **skeptical peer** who reviews measurement sections for a living

A measurement experiment can report a confident number that is mostly an
**artifact**, a value produced by the setup rather than by the thing you meant to
measure. Two artifacts do most of the damage, and both were hiding in the sensor
puzzle.

- **Demand effect**: when people being measured guess what result you want and shift
  their answers toward it. Example: participants sense a study is "about kindness," so
  they rate themselves kinder than they act. A sensor has no wishes, but the humans
  running or responding to a measurement do.
- **Instrument effect**: when the measuring device or question itself moves the
  reading, apart from the thing being measured. Example: a survey scale that starts at
  "strongly agree" pulls answers upward, and a sensor with an un-zeroed offset reads
  high no matter what it faces.

Both threaten **construct validity**, the degree to which your number actually
estimates the concept you meant, not something adjacent. A measurement with high
construct validity moves when the true construct moves and stays put otherwise.
Example: a "reading comprehension" test has weak construct validity if it mostly
measures reading *speed*.

> **A question that often comes up here:** *"If my design is randomized, doesn't that
> already handle these artifacts?"* No. Randomization balances *who* is in each group.
> It does nothing about a demand effect that hits everyone the same way, or an
> instrument that reads high for every unit. Those biases survive randomization
> untouched, which is why you diagnose and design them out separately.

The next cell draws the artifact DAG so you can see why a randomized measurement can
still be wrong. Three arrows feed the measured response. Only one of them is the
construct you want.

In [ ]:
# The artifact DAG: only one of three arrows into Y is the construct you want.
Ga = nx.DiGraph()
posa = {
    "Construct\ntheta":        (0.0, 1.4),
    "Demand\neffect (D)":      (0.0, 0.0),
    "Instrument\neffect (Q)":  (0.0, 2.8),
    "Measured\nresponse (Y)":  (2.6, 1.4),
}
honest_edge   = [("Construct\ntheta", "Measured\nresponse (Y)")]
artifact_edges = [("Demand\neffect (D)", "Measured\nresponse (Y)"),
                  ("Instrument\neffect (Q)", "Measured\nresponse (Y)")]
Ga.add_edges_from(honest_edge + artifact_edges)

fig, ax = plt.subplots(figsize=(9.5, 5))
node_fc = {"Construct\ntheta": "#DCE6F1", "Demand\neffect (D)": "#F4CCCC",
           "Instrument\neffect (Q)": "#F4CCCC", "Measured\nresponse (Y)": "#EAEAEA"}
node_ec = {"Construct\ntheta": "#4C72B0", "Demand\neffect (D)": "#C44E52",
           "Instrument\neffect (Q)": "#C44E52", "Measured\nresponse (Y)": "#888888"}
for n in posa:
    nx.draw_networkx_nodes(Ga, posa, nodelist=[n], node_color=node_fc[n],
                           edgecolors=node_ec[n], linewidths=2.0, node_size=6400, ax=ax)
nx.draw_networkx_labels(Ga, posa, font_size=8.5, ax=ax)
# The honest arrow is solid; the artifact arrows are dashed AND labeled, so the
# distinction is not carried by color alone.
nx.draw_networkx_edges(Ga, posa, edgelist=honest_edge, arrowstyle="-|>", arrowsize=18,
                       edge_color="#4C72B0", width=2.4, style="solid", node_size=6400, ax=ax)
nx.draw_networkx_edges(Ga, posa, edgelist=artifact_edges, arrowstyle="-|>", arrowsize=18,
                       edge_color="#C44E52", width=1.8, style="dashed", node_size=6400, ax=ax)
ax.text(1.3, 1.62, "wanted: the construct", color="#4C72B0", fontsize=8.5, fontweight="bold")
ax.text(1.15, 0.55, "artifact (dashed)", color="#C44E52", fontsize=8.5)
ax.text(1.15, 2.28, "artifact (dashed)", color="#C44E52", fontsize=8.5)
ax.set_title("Y = theta + demand + instrument: only the solid arrow is the construct")
ax.set_xlim(-1.2, 4.0)
ax.set_ylim(-0.7, 3.6)
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Artifact DAG drawn.")
print("  The measured response Y is fed by three arrows: the construct (solid) plus")
print("  two artifacts (dashed): a demand effect and an instrument effect.")
print("  A redesign that cuts the dashed arrows leaves Y measuring theta alone.")

**Reading the diagram.** The measured response is a sum, not a signal. The solid
arrow is the construct you want. The two dashed arrows are the artifacts that ride
along. If you report the measured response as if only the solid arrow fed it, you hand
your reader a number inflated or deflated by things that have nothing to do with the
construct. The sensor puzzle now has an answer: the third source of disagreement was
the **instrument effect**, each lab's offset, not the world.

*In plain terms, the picture says a measurement can be precise, repeatable, and still
measure the wrong thing, because the setup itself is one of the ingredients.*

### 🔮 Pause & Predict

You are about to run a measurement experiment where the construct's **true value is
5.0** on a 0-to-10 scale. A naive design lets participants guess the hypothesis (a
demand effect) and uses a leading scale (an instrument effect). Before running, commit:
will the naive measurement land above, at, or below the true 5.0, and roughly how far
off? Then predict which single redesign move, blinding the participants or neutralizing
the scale, buys back more of the truth. Write one sentence. Do not peek.

### YOUR ANSWER HERE:

**Naive measurement vs true 5.0 (direction and rough size):**

**Which redesign move buys back more truth, and why:**

---

### 🛠️ Hands-On: declare, diagnose, redesign a measurement experiment

Run the cell. It sets a true construct value, runs a **naive** measurement carrying a
demand effect and an instrument effect, diagnoses how far the number sits from the
truth, then **redesigns** by blinding participants and neutralizing the scale. Read the
three numbers before you read the next markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting whether the redesign lands the
measurement back near 5.0, or only part of the way.

> 💡 **Gemini Prompt:** "Here is a simulation that measures a construct three ways: the
> true value, a naive design with a demand effect and an instrument effect, and a
> redesign that removes both: [paste the next cell]. Explain, step by step, which line
> introduces the demand effect and which introduces the instrument effect, and confirm
> that the redesign removes exactly those two terms. Then propose one additional
> redesign move I did not include that would further protect construct validity."
>
> **After running, verify (counters *plausible-but-wrong-method*: a redesign that sounds
> principled can leave the artifact untouched if it targets the wrong term):**
> - [ ] Confirm the naive and redesigned numbers Gemini discusses match the values your
>   cell printed (naive near 7.0, redesigned near 5.0), and that the bias it names
>   decomposes into demand plus instrument, not some invented third term.
> - [ ] Check its extra redesign move against the DAG: does it actually cut a dashed
>   arrow (an artifact), or does it just add noise. A move that touches neither arrow
>   does not protect construct validity, so reject it.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Declare -> diagnose -> redesign a measurement experiment (seed 464).
rng_meas = np.random.default_rng(SEED)
N = 1200
TRUE_MU     = 5.0     # the TRUE value of the construct on a 0-to-10 scale
DEMAND      = 1.2     # participants shift UP toward the guessed hypothesis
INSTRUMENT  = 0.8     # a leading scale reads UP regardless of the construct
NOISE       = 2.0

# DECLARE: we want to measure TRUE_MU. Y = construct + demand + instrument + noise.
naive      = TRUE_MU + DEMAND + INSTRUMENT + rng_meas.normal(0, NOISE, N)
# REDESIGN: blind participants (cuts demand) + neutral scale (cuts instrument).
redesigned = TRUE_MU + rng_meas.normal(0, NOISE, N)

naive_est, redes_est = naive.mean(), redesigned.mean()
bias = naive_est - TRUE_MU

print("DECLARE : measure a construct whose true value is", TRUE_MU, "on a 0-to-10 scale.")
print("DIAGNOSE: run the naive design (demand + instrument both active):")
print(f"          naive measurement      = {naive_est:.2f}")
print(f"          bias above the truth    = {bias:+.2f}   "
      f"(demand {DEMAND:+.1f} + instrument {INSTRUMENT:+.1f} = {DEMAND+INSTRUMENT:+.1f})")
print("REDESIGN: blind participants (cut demand) + neutral scale (cut instrument):")
print(f"          redesigned measurement = {redes_est:.2f}   "
      f"(off the truth by {redes_est - TRUE_MU:+.2f})")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["Naive\n(demand+instrument)", "Redesigned\n(blinded+neutral)"],
              [naive_est, redes_est], color=["#C44E52", "#4C72B0"], edgecolor="white")
bars[0].set_hatch("//")   # a second channel beyond color for the biased bar
ax.axhline(TRUE_MU, color="#55A868", linestyle="--", linewidth=2.2,
           label=f"true construct value = {TRUE_MU:.1f}")
ax.set_ylim(0, 9)
ax.set_ylabel("Measured value of the construct (0 to 10)")
ax.set_title("Declare, diagnose, redesign: the redesign lands the measurement back on the truth")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the naive design overshoots; the redesign lands on the truth.
assert naive_est > TRUE_MU + 1.5, "naive design should read well above the truth"
assert abs(redes_est - TRUE_MU) < 0.4, "redesigned measurement should land near the truth"
assert naive_est - redes_est > 1.5, "the redesign should remove most of the artifact"
print(f"✓ Self-check passed: naive reads {naive_est:.2f} (bias {bias:+.2f}); "
      f"redesigned reads {redes_est:.2f}, within {abs(redes_est - TRUE_MU):.2f} of the true 5.0.")
print("  Same construct. The artifacts, not the world, moved the naive number.")

### 🔍 Reading the Evidence: what the redesign bought

The naive design reported about 7.0 for a construct whose true value is 5.0. The whole
gap was an artifact: a demand effect of about 1.2 plus an instrument effect of about
0.8. The redesign, which blinded participants and neutralized the scale, landed near
5.0. Notice what randomization alone would never have fixed. Both artifacts hit every
unit, so no amount of balancing groups would remove them.

In the cell below, write three things. First: the false headline the naive number
invites. Second: which artifact you would prioritize designing out if you could only
fix one, and why. Third: one sentence connecting this to **construct validity**, naming
what the naive design was really measuring instead of the construct.

### YOUR ANSWER HERE:

**The false headline the naive number invites:**

**The artifact I would design out first, and why:**

**What the naive design was really measuring (construct-validity sentence):**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 2's in-class block.** Everything below deepens
the same ideas and feeds your M6 protocol and your URC abstract. Finish it before
Friday's studio: the measurement-bench reading and its practice drill, the prompt you
adapt to your own project, the interrogation, the AI-free reach decision, the project
transfer, the ledger, and the Exit Defense. The 🧑‍⚖️ Human-Only Checkpoint is the
never-delegate core of M6, so give it real time.

---

## 5. The Measurement Bench: Conjoint, Behavioral Games, and STEM Instruments

**Guiding question:** *what other experiments measure, and which artifact each one has
to design out?*

The measurement idea is bigger than surveys. Meet five members of the bench, each a
randomized-stimulus design whose job is to *describe* a latent quantity.

- **Conjoint experiment**: randomly varies several attributes of a profile at once and
  measures how much each attribute shifts a stated choice, to describe which attributes
  people weigh. Example: varying a job's salary, remote option, and title to measure
  how much each moves acceptance.
- **Behavioral game**: a small structured decision with real stakes, used to measure a
  trait through choices rather than self-report. Example: a trust game where how much
  money you send a stranger measures your trust.
- **Calibration protocol**: repeatedly measuring a known reference standard to describe
  an instrument's bias and precision. Example: reading a certified 100.0 g mass ten
  times to see how far a scale drifts.
- **A/B instrument test**: randomly assigning two measurement procedures to the same
  units to describe how much the procedure itself moves the reading. Example: running
  the same blood samples through two analyzers to quantify the between-machine gap.
- **Psychophysics**: varying a physical stimulus in steps to measure the latent
  threshold at which people detect or discriminate it. Example: dimming a tone until
  you find the faintest volume a listener can still hear.

Every one of these can be faked by a demand effect, an instrument effect, or both. The
skill is naming which artifact each design most has to fear, and the redesign that
defuses it.

> **A question that often comes up here:** *"A conjoint or a trust game feels like it
> must be causal, since you are varying things and watching behavior change."* The
> variation is real, but the target is descriptive: how much each attribute *weighs*,
> or how much trust a person *holds*. You are measuring a standing preference or trait,
> not estimating the effect of a policy you would deploy. Same lesson as the audit
> study, wearing new clothes.

### 📝 Practice: name the artifact, name the redesign

*(Human-first: work all three yourself before any AI. Matching an artifact to a design
is the graded skill.)*

For each measurement design below, name the **demand effect or instrument effect** most
likely to fake its result, and the one **redesign** that defuses it. Answer in the cell
that follows.

- **A. Calibration protocol.** A lab measures a certified reference mass to report its
  scale's accuracy, but always weighs it first thing after the scale powers on.
- **B. A/B instrument test.** A team compares two survey wordings for "how stressed are
  you," and the same interviewer administers both, knowing which wording they prefer.
- **C. Conjoint task.** Respondents rate job profiles, and every profile that includes
  "remote work" also happens to be shown in a brighter, larger font.

### YOUR ANSWER HERE:

**A (artifact + redesign):**

**B (artifact + redesign):**

**C (artifact + redesign):**

---

### 🔁 Modify the Prompt

Above, you had Gemini explain the artifact simulation. Now turn the same partner on
*your* project. First, in one or two sentences, write the construct your project needs
to measure and one artifact you fear could fake it. Do not let AI draft this. Then adapt
the prompt below so Gemini red-teams *your* measurement plan.

> **Base prompt (the red-team):** "Here is a construct I plan to measure and how I plan
> to measure it: '[your plan].' Act as a hostile measurement reviewer. List every demand
> effect and instrument effect that could produce my result even if the true construct
> value were different. For each, name the redesign that would rule it out."

Rewrite it so it names *your* construct, *your* instrument, and *your* respondents or
device. In the cell below: paste your plan, paste your adapted prompt, predict the single
strongest artifact Gemini will name, then run it and note whether your prediction held.

**🏠 Homework depth: run this on your own.**

> 💡 **Gemini Prompt:** "[your adapted red-team prompt, naming your construct, your
> instrument, and your respondents]"
>
> **After running, verify (counters *illusion of completeness*: a long, tidy artifact
> list that omits the one that would actually sink your measurement):**
> - [ ] Compare the list against the artifact you committed to above. If yours is
>   missing, the tidy output failed the one test it looked complete on, so add it back.
> - [ ] Ask the follow-up: "what is the single most important artifact you left out, and
>   how would I detect it in my data?" Then judge that answer yourself against your design.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### YOUR ANSWER HERE:

**My construct and the artifact I fear (AI did not draft this):**

**My adapted red-team prompt (my construct, my instrument, my respondents):**

**The strongest artifact I predict Gemini will name:**

**What Gemini actually named, and what I added or rejected:**

---

### 🔬 Interrogate the Output

Gemini just handed you a list of artifacts threatening *your* measurement. Do not accept
the list as automatically right or complete. Interrogate it against four checks, and
answer each in the cell below.

- **Claims:** does each artifact it names actually apply to your design, or did it
  describe a generic survey problem your setup does not have?
- **Assumptions:** what does it assume about your respondents or instrument? Is that
  assumption yours to confirm, or one it invented?
- **Missing information:** which real artifact did it *miss*? Compare against the one you
  committed to, and against the demand and instrument arrows in the DAG.
- **Overstatements:** does any item sound more certain or more fatal than the evidence
  supports? Flag the exact words.

And one rule that governs the whole notebook: **code running without errors is not the
same as code being correct.** A measurement cell can execute cleanly and still compute
an artifact-laden number. Verify the *quantity*, not just the green check.

### YOUR ANSWER HERE:

**Claims (does each named artifact fit my design?):**

**Assumptions (did it invent a respondent or instrument detail?):**

**Missing information (the artifact it skipped):**

**Overstatements (exact words):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini for this one. No AI, no notebook search. This is a never-delegate decision:
whether your measurement inquiry is descriptive or causal, and which artifact you must
rule out, are yours to declare and defend. In the cell below, write, in your own words:

1. **Your project's measurement inquiry**, stated as a descriptive quantity (a level, a
   prevalence, a weight, a threshold) for named units. If the honest version is causal
   instead, say so, and note that a later week supplies that pathway.
2. **The one artifact you must rule out**, named as a demand effect or an instrument
   effect, plus the concrete redesign move that rules it out.
3. **Your one-sentence claim boundary:** the descriptive claim your design licenses,
   and the causal or population sentence you are forbidding yourself to write.

If you cannot yet state the descriptive quantity cleanly, that is a finding, not a
failure. It tells you your inquiry is not yet declared.

### YOUR ANSWER HERE:

**1. My measurement inquiry (descriptive quantity + units):**

**2. The one artifact I must rule out (demand or instrument) + the redesign:**

**3. My one-sentence claim boundary (licensed claim + forbidden sentence):**

---

### 🎯 Project Transfer: the spine of your M6 protocol

*(Human-first: draft every piece yourself. AI may critique it after, but the protocol
you defend at the studio has to be reasoned by you.)*

This is where the week comes together into your **M6 experimental measurement or
data-acquisition protocol**. In the cell below, draft the protocol's spine for *your*
project:

1. **The construct and the design:** the latent characteristic you must measure, and
   whether an experiment (audit, list, conjoint, behavioral game, calibration, A/B
   instrument test) or a direct measure serves it better, with one sentence of why.
2. **The descriptive quantity:** the exact number the design reports (a level, a
   prevalence, a weight, a threshold) and the units it describes.
3. **The artifact plan:** the demand effect and the instrument effect your design most
   fears, and the redesign move guarding each, tied to a dashed arrow in the DAG. If
   your project acquires data by another route, such as a scrape, an archive, or a
   survey pull, name the acquisition threats in the same shape: a **source that
   self-selects** and a **field that means something other than your construct**, each
   with the move that guards it.
4. **The boundary line (two sentences):** first, the descriptive claim your design
   licenses with its uncertainty; second, the causal or population sentence you forbid.

These four pieces are what you will walk a skeptic through at Friday's studio, and the
same pieces feed the URC abstract that clears its internal gate there. Write them so a
reader who has never seen your project could locate the exact edge of your measurement.

### YOUR ANSWER HERE:

**1. The construct and the design (with why):**

**2. The descriptive quantity and its units:**

**3. The artifact plan (demand + instrument + the redesign guarding each):**

**4. My boundary line, sentence 1 (licensed descriptive claim) and sentence 2 (forbidden causal/population sentence):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in as a
model, and it models the discipline this week teaches: **you decide the kind of inquiry
and the artifacts; the AI drafts and critiques, and you verify.** This notebook offers
five prompts, one live per lecture and three homework-depth, so your ledger carries a
row for each one you actually ran, not a fixed count. The ledger is a graded habit, not
paperwork: it is how you show your measurement was verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Explain the declare-diagnose-redesign simulation and propose one more construct-validity fix | Gemini | "Explain which line adds the demand effect and which adds the instrument effect, confirm the redesign removes both, and propose one more fix." | Correctly located both artifact terms; proposed adding a second neutral scale as a cross-check | Kept the cross-check idea; confirmed the redesigned mean matched my printout (near 5.0) | Read naive and redesigned means off my own output; checked the bias decomposed into demand + instrument | A real study will not hand me the true value to check the redesign against | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #06 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, drawn from today's measurements or
   your own project, that you would put your name on.
2. **Its boundary:** what this evidence does NOT establish. Name the compass position
   (description) and state that random assignment here did not license a causal
   inquiry.
3. **My uncertainty and limitations:** one sentence naming your interval, your worst
   unresolved artifact, or the resample or simulation caveat.
4. **AI check:** what you delegated to Gemini, whether you **accepted, changed, or
   rejected** what it returned, and the specific check that decided it (a number you
   recomputed, a source you retrieved, an artifact you confirmed).

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (description; random assignment did not buy a causal inquiry):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 6. Wrap-Up

Across two lectures you turned an experiment into a measuring instrument. You saw that a
randomized stimulus can serve a purely **descriptive** inquiry, because the kind of a
question lives in its words, not in the coin flip of its procedure. You read a real
survey experiment as a measurement of a crystallized attitude, recovered a hidden
prevalence with a list experiment that a direct question could not reach, and then met
the harder truth: a measurement is only as good as its freedom from demand and
instrument effects. You declared, diagnosed, and redesigned a measurement experiment,
and watched a number built mostly of artifacts fall back onto the construct it was
supposed to report.

> **"Random assignment measures; it does not automatically explain. An experiment
> earns the word 'causal' only when the question asks what an intervention would change.
> When the target is a latent characteristic, you have measured something, and the
> honest report describes it, with its artifacts ruled out and its uncertainty stated."**

Next week the compass turns to **prediction**: descriptive questions about unseen cases,
where the test is beating an honest baseline on data you have not looked at, and knowing
when a win is fake. Your M6 protocol presents at Friday's studio, where your URC abstract
clears its internal gate, and M7, the declared analysis protocol, kicks off. Bring your
construct, your descriptive quantity, and your artifact plan. This notebook companions
RDSS ch. 17, 'Experimental: descriptive.'

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 17 'Experimental: descriptive' | experiments as measurement systems, latent characteristics, audit/list/conjoint experiments, behavioral games, construct validity, demand and instrument effects, why assignment does not imply a causal inquiry | adapted (concept-level, honors non-quant audience)*
- *DeclareDesign experimental-descriptive design library | the experimental descriptive pathway framing (Model + Inquiry + Data strategy for a measurement inquiry) | referenced*
- *bonilla_tillery.csv | rdss package data | the four randomized identity frames read as a measurement of a crystallized attitude (level + frame-stability), with per-frame standard errors | adapted (analyzed descriptively, not as the original causal framing test)*
- *fresh | the list-experiment prevalence simulation + the declare-diagnose-redesign measurement simulation with demand and instrument effects (seed 464) | newly-constructed-from-source-concept*
- *fresh | the measurement-experiment DAG and the artifact DAG | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 17 'Experimental: descriptive'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- Bonilla, T., & Tillery, A. B. (2020). Which identity frames boost support for and
  mobilization in the Black Lives Matter movement? An experimental test. *American
  Political Science Review* (the survey experiment replicated in `bonilla_tillery`).

---

<center>

Thank you!

</center>